# CareLoop Clinics — No-Show Guard
## Part 3: Machine Learning

Prerequisite: run `01_ingest_clean.ipynb` and generate `appointments_clean.csv`.
This notebook evaluates logistic regression, decision trees and random forest,
selects a model using training-only cross-validation, tunes the threshold using
out-of-fold training predictions, and evaluates once on the held-out test set.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

def run_part_3_ml(clean_csv="appointments_clean.csv"):
    df = pd.read_csv(clean_csv)
    appt_ids = df["appointment_id"].copy()

    # ==========================================
    # Task 3.1 — Features, split and baseline
    # ==========================================
    feature_cols = [
        "age", "distance_km", "fee_inr", "lead_days",
        "prior_no_shows", "reminder_sent", "booking_channel", "visit_type"
    ]
    X = df[feature_cols].copy()
    X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
    X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
    y = df["no_show"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    baseline_acc = (y_test == 0).mean()
    print(f"Task 3.1 - X shape: {X.shape} (Target cols: 11)")
    print(f"Task 3.1 - Train rows: {len(X_train)}, Test rows: {len(X_test)}")
    print(f"Task 3.1 - Test no-shows: {y_test.sum()} (Target: 49)")
    print(f"Task 3.1 - Baseline accuracy (always Attended): {baseline_acc:.4f} (Target: 0.7525)")

    # ==========================================
    # Task 3.2 — Logistic regression
    # ==========================================
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train_s, y_train)
    lr_preds = lr.predict(X_test_s)
    lr_probs = lr.predict_proba(X_test_s)[:, 1]

    lr_acc = accuracy_score(y_test, lr_preds)
    lr_prec = precision_score(y_test, lr_preds)
    lr_rec = recall_score(y_test, lr_preds)
    lr_f1 = f1_score(y_test, lr_preds)
    lr_auc = roc_auc_score(y_test, lr_probs)

    print("\n--- Task 3.2: Logistic Regression (Test Set) ---")
    print(f"Accuracy: {lr_acc:.4f} | Precision: {lr_prec:.4f} | Recall: {lr_rec:.4f} | F1: {lr_f1:.4f} | ROC-AUC: {lr_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, lr_preds))

    coef_df = pd.DataFrame({
        "feature": X.columns,
        "coef": lr.coef_[0],
        "scale": scaler.scale_
    })
    print("\nCoefficients:\n", coef_df[["feature", "coef"]].to_string(index=False))

    rem_row = coef_df[coef_df["feature"] == "reminder_sent"].iloc[0]
    odds_scaled = np.exp(rem_row["coef"])
    odds_unscaled = np.exp(rem_row["coef"] / rem_row["scale"])
    print(f"\nreminder_sent -> exp(coef) [1 SD]: {odds_scaled:.4f} (Target: ~0.56)")
    print(f"reminder_sent -> exp(coef / scale) [No->Yes]: {odds_unscaled:.4f} (Target: ~0.31, ~69% lower odds)")

    # ==========================================
    # Task 3.3 — Decision tree and overfitting
    # ==========================================
    dt_full = DecisionTreeClassifier(random_state=42)
    dt_full.fit(X_train, y_train)
    print("\n--- Task 3.3: Decision Trees ---")
    print(f"Full Tree (depth={dt_full.get_depth()}) -> Train Acc: {dt_full.score(X_train, y_train):.4f}, Test Acc: {dt_full.score(X_test, y_test):.4f}")

    dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
    dt_3.fit(X_train, y_train)
    dt3_preds = dt_3.predict(X_test)
    dt3_probs = dt_3.predict_proba(X_test)[:, 1]
    dt3_acc = accuracy_score(y_test, dt3_preds)
    dt3_rec = recall_score(y_test, dt3_preds)
    dt3_auc = roc_auc_score(y_test, dt3_probs)
    print(f"Depth-3 Tree -> Train Acc: {dt_3.score(X_train, y_train):.4f}, Test Acc: {dt3_acc:.4f}")

    # ==========================================
    # Task 3.4 — Random forest & comparison table
    # ==========================================
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)
    rf_probs = rf.predict_proba(X_test)[:, 1]
    rf_acc = accuracy_score(y_test, rf_preds)
    rf_rec = recall_score(y_test, rf_preds)
    rf_auc = roc_auc_score(y_test, rf_probs)

    imp_df = pd.DataFrame({"feature": X.columns, "importance": rf.feature_importances_}).sort_values("importance", ascending=False)
    print("\n--- Task 3.4: Random Forest & Model Comparison ---")
    print("Top 3 RF Feature Importances:\n", imp_df.head(3).to_string(index=False))

    comp_df = pd.DataFrame([
        {"Model": "Logistic Regression", "Accuracy": round(lr_acc, 4), "Recall": round(lr_rec, 4), "ROC-AUC": round(lr_auc, 4)},
        {"Model": "Decision Tree (depth=3)", "Accuracy": round(dt3_acc, 4), "Recall": round(dt3_rec, 4), "ROC-AUC": round(dt3_auc, 4)},
        {"Model": "Random Forest (100)", "Accuracy": round(rf_acc, 4), "Recall": round(rf_rec, 4), "ROC-AUC": round(rf_auc, 4)},
    ])
    print("\nTest Comparison Table (For Reporting Only):\n", comp_df.to_string(index=False))

    # ==========================================
    # Task 3.5 — Tune, choose model & threshold, test once
    # ==========================================
    grid = GridSearchCV(
        RandomForestClassifier(random_state=42),
        {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
        cv=5,
        scoring="roc_auc"
    )
    grid.fit(X_train, y_train)
    print("\n--- Task 3.5: CV Model Selection & Threshold Tuning ---")
    print(f"(a) Tuned RF best_params_: {grid.best_params_} | best_score_ (CV AUC): {grid.best_score_:.4f}")

    lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
    cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
    cv_dt3 = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
    cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
    print(f"(b) 5-Fold CV ROC-AUC -> LR: {cv_lr:.4f} | DT3: {cv_dt3:.4f} | RF: {cv_rf:.4f} | Tuned RF: {grid.best_score_:.4f}")

    # Out-of-fold probabilities on training set for winner (Logistic Regression)
    oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
    print("\n(c) Out-of-Fold Threshold Comparison (Training Set):")
    for thresh in [0.5, 0.35]:
        oof_flag = (oof_probs >= thresh).astype(int)
        p = precision_score(y_train, oof_flag)
        r = recall_score(y_train, oof_flag)
        f1 = f1_score(y_train, oof_flag)
        print(f"    Threshold {thresh:.2f} -> Precision: {p:.4f}, Recall: {r:.4f}, F1: {f1:.4f}, Flagged: {oof_flag.sum()}")

    # Final single evaluation on test set at chosen threshold 0.35
    chosen_thresh = 0.35
    lr_pipe.fit(X_train, y_train)
    test_probs = lr_pipe.predict_proba(X_test)[:, 1]
    test_flags = (test_probs >= chosen_thresh).astype(int)

    print("\n(d) Final Test Evaluation (Logistic Regression @ threshold 0.35):")
    print(f"    Precision: {precision_score(y_test, test_flags):.4f}")
    print(f"    Recall:    {recall_score(y_test, test_flags):.4f}")
    print(f"    Flagged:   {test_flags.sum()} / {len(y_test)}")
    print(f"    ROC-AUC:   {roc_auc_score(y_test, test_probs):.4f}")

    # (e) Save test_risk_scores.csv
    scores_df = pd.DataFrame({
        "appointment_id": appt_ids.loc[X_test.index].values,
        "no_show": y_test.values,
        "risk_prob": np.round(test_probs, 4),
        "flag": test_flags
    })
    scores_df.to_csv("test_risk_scores.csv", index=False)
    print(f"\n(e) Saved test_risk_scores.csv ({len(scores_df)} rows, {scores_df['flag'].sum()} flagged).")
    return scores_df

scores_df = run_part_3_ml()

Task 3.1 - X shape: (990, 11) (Target cols: 11)
Task 3.1 - Train rows: 792, Test rows: 198
Task 3.1 - Test no-shows: 49 (Target: 49)
Task 3.1 - Baseline accuracy (always Attended): 0.7525 (Target: 0.7525)



--- Task 3.2: Logistic Regression (Test Set) ---
Accuracy: 0.7879 | Precision: 0.6296 | Recall: 0.3469 | F1: 0.4474 | ROC-AUC: 0.7889
Confusion Matrix:
 [[139  10]
 [ 32  17]]

Coefficients:
                 feature      coef
                    age -0.253880
            distance_km  0.253448
                fee_inr  0.452761
              lead_days  0.614471
         prior_no_shows  0.649612
          reminder_sent -0.580669
    booking_channel_App -0.046804
  booking_channel_Phone  0.215881
booking_channel_Walk-In -0.186364
visit_type_Consultation  0.015658
   visit_type_Follow-up -0.015658

reminder_sent -> exp(coef) [1 SD]: 0.5595 (Target: ~0.56)
reminder_sent -> exp(coef / scale) [No->Yes]: 0.3094 (Target: ~0.31, ~69% lower odds)

--- Task 3.3: Decision Trees ---
Full Tree (depth=15) -> Train Acc: 1.0000, Test Acc: 0.6717
Depth-3 Tree -> Train Acc: 0.8220, Test Acc: 0.7778



--- Task 3.4: Random Forest & Model Comparison ---
Top 3 RF Feature Importances:
     feature  importance
  lead_days    0.219905
        age    0.196788
distance_km    0.194739

Test Comparison Table (For Reporting Only):
                   Model  Accuracy  Recall  ROC-AUC
    Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (depth=3)    0.7778  0.2245   0.6997
    Random Forest (100)    0.7475  0.3061   0.7057



--- Task 3.5: CV Model Selection & Threshold Tuning ---
(a) Tuned RF best_params_: {'max_depth': 3, 'n_estimators': 200} | best_score_ (CV AUC): 0.7673


(b) 5-Fold CV ROC-AUC -> LR: 0.7737 | DT3: 0.6784 | RF: 0.7466 | Tuned RF: 0.7673

(c) Out-of-Fold Threshold Comparison (Training Set):
    Threshold 0.50 -> Precision: 0.5753, Recall: 0.2485, F1: 0.3471, Flagged: 73
    Threshold 0.35 -> Precision: 0.5130, Recall: 0.4675, F1: 0.4892, Flagged: 154

(d) Final Test Evaluation (Logistic Regression @ threshold 0.35):
    Precision: 0.5714
    Recall:    0.5714
    Flagged:   49 / 198
    ROC-AUC:   0.7889



(e) Saved test_risk_scores.csv (198 rows, 49 flagged).
